# Notebook 01: Data Profiling
**Objective:** Understand the structure, distributions, and quality of the deal desk dataset before analysis.

Tables: `deals`, `approvals`, `outcomes`

In [1]:
import pandas as pd
import numpy as np
import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Load CSVs
deals_df = pd.read_csv('../data/raw/deals.csv')
approvals_df = pd.read_csv('../data/raw/approvals.csv')
outcomes_df = pd.read_csv('../data/raw/outcomes.csv')

# Register with DuckDB
con = duckdb.connect()
con.register('deals', deals_df)
con.register('approvals', approvals_df)
con.register('outcomes', outcomes_df)

print(f"deals:     {len(deals_df):,} rows, {deals_df.shape[1]} cols")
print(f"approvals: {len(approvals_df):,} rows, {approvals_df.shape[1]} cols")
print(f"outcomes:  {len(outcomes_df):,} rows, {outcomes_df.shape[1]} cols")

deals:     2,000 rows, 10 cols
approvals: 660 rows, 8 cols
outcomes:  2,000 rows, 5 cols


## Schema Review

In [3]:
for name, df in [('deals', deals_df), ('approvals', approvals_df), ('outcomes', outcomes_df)]:
    print(f"\n--- {name} ---")
    print(df.dtypes)


--- deals ---
deal_id          object
rep_id           object
account_id       object
segment          object
arr               int64
list_price        int64
discount_pct    float64
deal_type        object
product          object
created_date     object
dtype: object

--- approvals ---
approval_id          object
deal_id              object
approver             object
submitted_date       object
decision_date        object
cycle_days          float64
status               object
rejection_reason     object
dtype: object

--- outcomes ---
deal_id        object
stage          object
close_date     object
win_flag        int64
final_arr     float64
dtype: object


## Null Check

In [4]:
for name, df in [('deals', deals_df), ('approvals', approvals_df), ('outcomes', outcomes_df)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"\n--- {name} ---")
    print(nulls if len(nulls) > 0 else "no nulls")


--- deals ---
no nulls

--- approvals ---
decision_date        35
cycle_days           35
rejection_reason    576
dtype: int64

--- outcomes ---
no nulls


## Date Casting

In [5]:
deals_df['created_date'] = pd.to_datetime(deals_df['created_date'])
approvals_df['submitted_date'] = pd.to_datetime(approvals_df['submitted_date'])
approvals_df['decision_date'] = pd.to_datetime(approvals_df['decision_date'])
outcomes_df['close_date'] = pd.to_datetime(outcomes_df['close_date'])

print("dates cast successfully")

dates cast successfully


## Distribution Overview

In [6]:
for col in ['segment', 'deal_type', 'product']:
    print(f"\n--- {col} ---")
    counts = deals_df[col].value_counts()
    pcts = deals_df[col].value_counts(normalize=True) * 100
    for val in counts.index:
        print(f"  {val:<20} {counts[val]:>5,}  ({pcts[val]:.1f}%)")


--- segment ---
  SMB                    851  (42.5%)
  Mid-Market             751  (37.5%)
  Enterprise             398  (19.9%)

--- deal_type ---
  New Business         1,017  (50.8%)
  Expansion              611  (30.6%)
  Renewal                372  (18.6%)

--- product ---
  Core                   808  (40.4%)
  Professional           694  (34.7%)
  Enterprise Suite       498  (24.9%)


## Discount Distribution

In [7]:
print(f"{'Segment':<15} {'Mean':>8} {'Median':>8} {'Min':>8} {'Max':>8} {'Std':>8}")
print("-" * 55)

segments = ['SMB', 'Mid-Market', 'Enterprise']
for seg in segments:
    d = deals_df[deals_df['segment'] == seg]['discount_pct']
    print(
        f"{seg:<15} {d.mean():>7.1%} {d.median():>7.1%} "
        f"{d.min():>7.1%} {d.max():>7.1%} {d.std():>7.1%}"
    )

d = deals_df['discount_pct']
print(
    f"\n{'Overall':<15} {d.mean():>7.1%} {d.median():>7.1%} "
    f"{d.min():>7.1%} {d.max():>7.1%} {d.std():>7.1%}"
)

Segment             Mean   Median      Min      Max      Std
-------------------------------------------------------
SMB                9.2%    9.2%    0.0%   26.2%    5.7%
Mid-Market        11.5%   11.6%    0.0%   29.4%    6.0%
Enterprise        14.6%   14.6%    0.0%   30.2%    5.8%

Overall           11.2%   11.2%    0.0%   30.2%    6.1%


## ARR Distribution

In [8]:
print(f"{'Segment':<15} {'Mean':>12} {'Median':>12} {'Min':>12} {'Max':>12}")
print("-" * 65)

segments = ['SMB', 'Mid-Market', 'Enterprise']
for seg in segments:
    d = deals_df[deals_df['segment'] == seg]['arr']
    print(
        f"{seg:<15} ${d.mean():>10,.0f} ${d.median():>10,.0f} "
        f"${d.min():>10,.0f} ${d.max():>10,.0f}"
    )

d = deals_df['arr']
print(
    f"\n{'Overall':<15} ${d.mean():>10,.0f} ${d.median():>10,.0f} "
    f"${d.min():>10,.0f} ${d.max():>10,.0f}"
)

Segment                 Mean       Median          Min          Max
-----------------------------------------------------------------
SMB             $    17,148 $    16,618 $     5,016 $    29,977
Mid-Market      $    89,510 $    90,233 $    30,005 $   149,735
Enterprise      $   483,491 $   481,468 $   150,209 $   796,290

Overall         $   137,122 $    52,904 $     5,016 $   796,290


## Approval Status Distribution

In [9]:
counts = approvals_df['status'].value_counts()
pcts = approvals_df['status'].value_counts(normalize=True) * 100
print(f"{'Status':<12} {'Count':>8} {'Pct':>8}")
print("-" * 30)
for val in counts.index:
    print(f"{val:<12} {counts[val]:>8,} {pcts[val]:>7.1f}%")

print(f"\nTotal deals requiring approval: {len(approvals_df):,} of {len(deals_df):,} ({len(approvals_df)/len(deals_df):.1%})")

Status          Count      Pct
------------------------------
Approved          541    82.0%
Rejected           84    12.7%
Pending            35     5.3%

Total deals requiring approval: 660 of 2,000 (33.0%)


## Win Rate Overview

In [10]:
merged = outcomes_df.merge(
    deals_df[['deal_id', 'segment', 'discount_pct']], on='deal_id'
)

print(f"{'Segment':<15} {'Deals':>8} {'Won':>8} {'Win Rate':>10} {'Avg Discount':>14}")
print("-" * 57)

segments = ['SMB', 'Mid-Market', 'Enterprise']
for seg in segments:
    d = merged[merged['segment'] == seg]
    print(
        f"{seg:<15} {len(d):>8,} {d['win_flag'].sum():>8,} "
        f"{d['win_flag'].mean():>9.1%} {d['discount_pct'].mean():>13.1%}"
    )

print(
    f"\n{'Overall':<15} {len(merged):>8,} {merged['win_flag'].sum():>8,} "
    f"{merged['win_flag'].mean():>9.1%} {merged['discount_pct'].mean():>13.1%}"
)

Segment            Deals      Won   Win Rate   Avg Discount
---------------------------------------------------------
SMB                  851      559     65.7%          9.2%
Mid-Market           751      416     55.4%         11.5%
Enterprise           398      178     44.7%         14.6%

Overall            2,000    1,153     57.6%         11.2%


## Key Findings

- 2,000 deals across 20 reps; SMB 42.5%, Mid-Market 37.5%, Enterprise 19.9%
- Average discount is 11.2%; Enterprise deals average 14.6%
- 33% of deals (660) triggered approval; 82.0% approved, 12.7% rejected
- Overall win rate is 57.6%; Enterprise wins least (44.7%) despite highest discounts
- No data quality issues; dates cast, nulls in approvals are expected (pending + non-rejections)